# How to Write a Custom Encoder

<a href="https://colab.research.google.com/github/quprep/quprep/blob/main/examples/how-to/write_a_custom_encoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/quprep/quprep/v0.10.0?labpath=examples%2Fhow-to%2Fwrite_a_custom_encoder.ipynb)
<a href="https://account.qbraid.com?gitHubUrl=https://github.com/quprep/quprep/blob/main/examples/how-to/write_a_custom_encoder.ipynb"><img src="https://qbraid-static.s3.amazonaws.com/logos/Launch_on_qBraid_white.png" height="23"></a>


QuPrep's plugin registry lets you register custom encoders without forking the package. Subclass `BaseEncoder`, decorate with `@register_encoder`, and it works everywhere built-in encoders do: `Pipeline`, `encoding_sensitivity()`, and the full metrics suite.

In [ ]:
!pip install -q quprep

In [ ]:
import warnings

import numpy as np

import quprep as qd
from quprep import QuPrepWarning
from quprep.encode.base import BaseEncoder, EncodedResult
from quprep.plugins import get_encoder_class, list_encoders, register_encoder, unregister_encoder

print(f"quprep {qd.__version__}")

## 1. Define and register a custom encoder

Subclass `BaseEncoder`. Implement `encode()`, `n_qubits`, and `depth`. Decorate with `@register_encoder("name")` — registration happens at class definition time.

In [ ]:
@register_encoder("hadamard_angle")
class HadamardAngleEncoder(BaseEncoder):
    """Applies H then Ry(x_i) per qubit — superposition before rotation."""

    name = "hadamard_angle"

    @property
    def n_qubits(self):
        return None  # data-dependent: one qubit per feature

    @property
    def depth(self):
        return 2  # H + Ry per qubit

    def encode(self, x: np.ndarray) -> EncodedResult:
        n = len(x)
        angles = x.tolist()

        def circuit_fn():
            lines = ["OPENQASM 3.0;", 'include "stdgates.inc";', f"qubit[{n}] q;"]
            for i, a in enumerate(angles):
                lines.append(f"h q[{i}];")
                lines.append(f"ry({a}) q[{i}];")
            return "\n".join(lines)

        return EncodedResult(
            circuit_fn=circuit_fn,
            metadata={"encoding": self.name, "n_qubits": n, "depth": 2},
            parameters={"angles": angles},
        )

print(f"Registered encoders : {list_encoders()}")

## 2. Use in a Pipeline

In [ ]:
rng = np.random.default_rng(0)
X = rng.uniform(0, np.pi, (5, 3))
ds = qd.NumpyIngester().load(X)

with warnings.catch_warnings():
    warnings.simplefilter("ignore", QuPrepWarning)
    result = qd.Pipeline(encoder=HadamardAngleEncoder()).fit_transform(ds)

print(f"Circuits : {len(result.encoded)}")
print(f"Qubits   : {result.encoded[0].metadata['n_qubits']}")
print(f"Depth    : {result.encoded[0].metadata['depth']}")

## 3. QASM output via circuit_fn()

In [ ]:
qasm = qd.QASMExporter().export(result.encoded[0])
print(qasm)

## 4. Retrieve encoder by name from registry

In [ ]:
EncoderCls = get_encoder_class("hadamard_angle")
enc_from_registry = EncoderCls()
sample = X[0]
encoded_sample = enc_from_registry.encode(sample)
print(f"Retrieved class : {EncoderCls.__name__}")
print(f"Sample qubits   : {encoded_sample.metadata['n_qubits']}")

## 5. Unregister

In [ ]:
unregister_encoder("hadamard_angle")
print(f"Registered encoders after cleanup : {list_encoders()}")